# POLSCI 3

## Going Further: Working with Dates using `lubridate`

**This notebook is optional and not tied to any particular week.** Do it whenever you have time. Dates show up in almost every real dataset (when was the survey taken? when did the law pass? how many days until the election?), and R doesn't automatically understand them. This notebook teaches you how to work with them.

### What you'll learn

| Tool | What it does |
|---|---|
| `ymd()`, `mdy()`, `dmy()` | Turn text like `"3/15/2019"` into a real date |
| `year()`, `month()`, `day()`, `wday()` | Pull the year, month, day, or weekday out of a date |
| Date math (`-`, `+ days()`, `today()`) | How many days apart are two dates? What is 30 days later? |
| Comparing dates, `between()` | Keep only the rows in a date range |
| `floor_date()` | Round dates down to the start of the week, month, or year |
| `case_when()` | Make categories (like before / during / after) from dates |
| `ymd_hm()`, `hour()` | Dates that also have a time of day |

### How it works

Each section has (1) an explanation, (2) **examples** you can run, and (3) **your turn** on real flight data from the Bay Area. After each exercise, run the check cell (`. = ottr::check(...)`) to see if you're right. Each exercise tells you what to produce and what to name it. You figure out the steps. This is not graded.

The multiple-choice checks will tell you if you're wrong, so you *could* just try every letter. But you'd only be cheating yourself: this is for your own learning, and you can always ask me for help.

In [ ]:
library(tidyverse)   # dplyr, ggplot2, and more
library(lubridate)   # the dates package (new in this notebook)

`lubridate` is its own package, so it gets its own `library()` line. (Some versions of `tidyverse` load it for you and some don't, so we always load it ourselves.)

### Loading the data

The flight data is stored in `.dta` files, one for each airport and year (10 files in all). The cell below combines them into one table called `flights`. **You can ignore the code in this cell.** Just run it.

In [ ]:
# You can ignore this code: it combines the .dta files into one table called flights
flights <- list.files('ps3_flights', pattern = '\\.dta$', full.names = TRUE) %>%
  map_dfr(haven::read_dta)

flights

### The data

`flights` has one row per flight to or from one of the Bay Area's two big airports: Oakland (**OAK**) and San Francisco (**SFO**). It covers **2019 through 2023**: before, during, and after the COVID-19 pandemic. The columns are:

- `flight_date`: the day the flight was scheduled to leave, written like `3/15/2019`
- `sched_departure`: the scheduled departure date *and* time, like `3/15/2019 14:30` (24-hour clock, local time where the plane departs from)
- `carrier`: the airline's two-letter code (UA = United, WN = Southwest, AS = Alaska, DL = Delta, AA = American, and so on)
- `origin` and `dest`: the airports the flight leaves from and flies to
- `airport`: which of the two Bay Area airports (OAK or SFO) the flight involves
- `direction`: `"departing"` if the flight leaves that Bay Area airport, `"arriving"` if it lands there
- `dep_delay` and `arr_delay`: minutes late leaving and arriving (negative means early; `NA` means the flight was cancelled or didn't report)
- `cancelled`: 1 if the flight was cancelled, 0 if not
- `distance`: miles

The data come from the U.S. Bureau of Transportation Statistics. It includes every **domestic** flight reported to them (no international flights), so each row is a real flight, not a sample.

Now look at the `<chr>` under `flight_date` and `sched_departure` above. That means R is treating them as plain **text**, even though they look like dates. Fixing that is the first job.

### A few real dates for the examples

For the examples we'll use four real COVID-era dates, so you can see every row. Run this cell and look at it.

In [ ]:
events <- tibble(
  event     = c("WHO declares COVID-19 a pandemic",
                "California issues a statewide stay-at-home order",
                "California fully reopens its economy",
                "A federal judge strikes down the travel mask mandate"),
  date_text = c("2020-03-11", "2020-03-19", "2021-06-15", "2022-04-18")
)
events

---
## 1. Creating dates from text

Right now `events$date_text` is just text. R can't tell you which of two text dates is later, or how many days are between them. In fact, text gets sorted alphabetically, which gives silly answers:

In [ ]:
class(flights$flight_date)                 # "character": just text, as far as R is concerned

# As text, "3/15/2019" is "bigger" than "12/1/2020" because "3" comes after "1" alphabetically
max(c("3/15/2019", "12/1/2020"))

`lubridate` fixes this with a family of functions named after the **order** of the pieces in your text: **y**ear, **m**onth, **d**ay. If your text is year-month-day, use `ymd()`. If it's month-day-year (how Americans usually write it), use `mdy()`. If it's day-month-year, use `dmy()`. You don't have to tell R what's *between* the pieces: dashes, slashes, spaces, even month names all work.

In [ ]:
ymd("2020-03-19")          # year-month-day
ymd("20200319")            # no separators at all? still fine
mdy("3/19/2020")           # month-day-year
mdy("March 19, 2020")      # month names work too
dmy("19/3/2020")           # day-month-year
dmy("19 March 2020")

These functions work on a whole column at once. Once your text is converted, R knows it's a **Date**, so it knows which date is later:

In [ ]:
dates <- mdy(c("3/15/2019", "12/1/2020"))
dates
class(dates)     # "Date"
max(dates)       # now the right answer: the later date

Under the hood, R stores a date as a number: the count of days since January 1, 1970. That's why R can sort dates and subtract them.

If you pick the wrong function, the order doesn't match and you get `NA` (and a warning telling you how many dates failed). If you ever see `failed to parse`, check that your function matches the order in your text.

In [ ]:
as.numeric(ymd("1970-01-02"))     # 1 day after the starting point

mdy("2020-03-19")                 # wrong function for this text: NA and a warning

To turn a text column into a date column inside a data frame, use `mutate()`, just like you've been doing:

In [ ]:
events %>%
  mutate(date = ymd(date_text))

(If you have the year, month, and day in **three separate columns**, `make_date(year, month, day)` puts them together: `make_date(2020, 3, 19)`.)

### Your turn

**Q1.** A column of dates is written with the **day first**, like `"19/03/2020"`. Which line converts it to a date correctly?

- **A.** `ymd("19/03/2020")`
- **B.** `mdy("19/03/2020")`
- **C.** `dmy("19/03/2020")`
- **D.** `year("19/03/2020")`

Save your choice as a letter (text in quotes) in `q1.answer`.

In [ ]:
q1.answer <- ""   # type your letter between the quotes

In [ ]:
. = ottr::check("tests/q1.R")

**Q2.** The dates in `flights$flight_date` are written like `3/15/2019`. Create `q2.flights`, which is `flights` plus a new column called `date` that has `flight_date` converted from text to a real date. (Look at how the dates are written, then pick the matching function.)

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q2.R")

**Q3.** Here are three of the dates from `events`, written the way a European newspaper might write them. Convert them to dates and save the result as `covid.dates`.

In [ ]:
covid.strings <- c("11 March 2020", "19 March 2020", "15 June 2021")

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q3.R")

We'll use the dates from Q3 as landmarks below: **March 19, 2020** (California's stay-at-home order) and **June 15, 2021** (California fully reopens).

Before moving on, run this cell. It does what you did in Q2, so that everyone starts the next sections with the same table, called `flights.dates`.

In [ ]:
flights.dates <- flights %>%
  mutate(date = mdy(flight_date))

---
## 2. Pulling out the year, month, and day

Once R knows something is a date, you can pull out any piece of it: `year()`, `month()`, `day()`. For the day of the week use `wday()`. For month names and day names instead of numbers, add `label = TRUE` (and `abbr = FALSE` for the full name).

In [ ]:
d <- ymd("2020-03-19")

year(d)
month(d)
day(d)

month(d, label = TRUE)                  # the month's name (abbreviated)
month(d, label = TRUE, abbr = FALSE)    # the full name

wday(d)                                 # day of the week as a number (1 = Sunday)
wday(d, label = TRUE)                   # the day's name
wday(d, label = TRUE, abbr = FALSE)

yday(d)                                 # day of the year (1 to 365)

These work inside `mutate()` to create new columns, and inside `group_by()` or `summarize()` to compare groups. Here is a table of the events with all the pieces pulled out:

In [ ]:
events %>%
  mutate(date    = ymd(date_text),
         year    = year(date),
         month   = month(date, label = TRUE),
         weekday = wday(date, label = TRUE))

And here is how you could count flights in each calendar **month** (January through December, all five years pooled together):

In [ ]:
flights.dates %>%
  group_by(month = month(date, label = TRUE)) %>%
  summarize(flights = n())

### Your turn

**Q4.** Create `q4.flights`, which is `flights.dates` plus three new columns: `year`, `month`, and `day`, each a **number** pulled from `date`. (Don't use `label = TRUE` here.)

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q4.R")

**Q5.** How many flights are in the data for each **year**? Create `flights.per.year`, with columns `year` and `flights` (5 rows, one per year). Look at your result: what happened in 2020?

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q5.R")

**Q6.** Which days of the week are busiest? Create `weekday.counts`, with columns `weekday` (the day's name, using `wday()` with `label = TRUE`) and `flights` (the number of flights on that day of the week). It should have 7 rows.

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q6.R")

---
## 3. Date math and comparing dates

Because dates are really numbers, you can do arithmetic with them.

- **Subtract** two dates to find how many days apart they are.
- **Add or subtract a length of time** with `days()`, `weeks()`, `months()`, or `years()`.
- `today()` gives today's date (so its answer changes every day!).

In [ ]:
# How many days between the WHO announcement and California's order?
ymd("2020-03-19") - ymd("2020-03-11")

# The same, but as a plain number (useful for calculations)
as.numeric(ymd("2020-03-19") - ymd("2020-03-11"))

# Adding and subtracting lengths of time
ymd("2020-03-19") + days(30)
ymd("2020-03-19") - weeks(2)
ymd("2020-03-19") + months(1)
ymd("2020-03-19") + years(1)

# Today's date, and how long ago California's order was
today()
today() - ymd("2020-03-19")

(One quirk: `ymd("2020-01-31") + months(1)` gives `NA`, because there is no February 31.)

You can also **compare** dates with `<`, `>`, `<=`, `>=`, and `==`, which means you can use `filter()` to keep a range of dates. `between(x, low, high)` is a shortcut for "at least `low` and at most `high`." Always compare a date column to a date made with `ymd()` (or `mdy()`), not to text.

In [ ]:
ymd("2020-03-19") < ymd("2021-06-15")     # TRUE: the order came first

events %>%
  mutate(date = ymd(date_text)) %>%
  filter(date >= ymd("2021-01-01"))                       # events from 2021 on

events %>%
  mutate(date = ymd(date_text)) %>%
  filter(between(date, ymd("2020-03-15"), ymd("2021-12-31")))   # events in a date range

# New columns from date math: days since the WHO announcement
events %>%
  mutate(date = ymd(date_text),
         days_after_who = as.numeric(date - ymd("2020-03-11")))

### Your turn

**Q7.** What does R return for `ymd("2021-06-15") - ymd("2020-03-19")`?

- **A.** An error, because you can't subtract dates.
- **B.** The number of days between the two dates (R labels it "Time difference of ... days").
- **C.** Another date, somewhere around the year 1.
- **D.** The number of months between the two dates.

Save your choice as a letter (text in quotes) in `q7.answer`.

In [ ]:
q7.answer <- ""   # type your letter between the quotes

In [ ]:
. = ottr::check("tests/q7.R")

**Q8.** Create `after.order`, containing the flights from `flights.dates` that happened from **March 19, 2020** (the stay-at-home order) up to **and including** the day that is **30 days later**. (That is 31 calendar days.)

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q8.R")

**Q9.** Create `flights.order`, which is `flights.dates` plus a new column `days_since_order`: the **number of days** between the flight's `date` and March 19, 2020 (so it's negative for flights before the order and positive for flights after). Make it a plain number, not a "Time difference."

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q9.R")

---
## 4. Rounding dates: `floor_date()`

Often you don't want to compare every single day. You want **weeks, months, or years**. `floor_date(date, "month")` rounds each date **down** to the first day of its month, so every date in March 2020 becomes `2020-03-01`. Then `group_by()` can treat them as a group. Other choices are `"week"` (rounds down to the Sunday that starts the week), `"year"`, and `"quarter"`. (`ceiling_date()` rounds up and `round_date()` rounds to the nearest.)

In [ ]:
floor_date(ymd("2020-03-19"), "month")
floor_date(ymd("2020-03-19"), "year")
floor_date(ymd("2020-03-19"), "week")

Here is how to count flights **per week** in March and April of 2020, right as the pandemic hit. (The last week is short because we stop at the end of April.)

In [ ]:
weekly.spring <- flights.dates %>%
  filter(date >= ymd("2020-03-01") & date < ymd("2020-05-01")) %>%
  mutate(week_start = floor_date(date, "week")) %>%
  group_by(week_start) %>%
  summarize(flights = n())

weekly.spring

Dates make great graphs. **`geom_line()`** connects the points in order from left to right, which is what you want for something that changes over time. When the x-axis is a date, `scale_x_date()` controls the labels: `date_breaks` says how often to put a tick mark (`"1 month"`, `"2 weeks"`, `"1 year"`) and `date_labels` says how to write the dates using codes: `%Y` is the year (2020), `%y` is the short year (20), `%b` is the short month name (Mar), `%B` is the full month name (March), `%m` is the month number, and `%d` is the day.

In [ ]:
ggplot(weekly.spring, aes(x = week_start, y = flights)) +
  geom_line() +
  geom_point() +
  scale_x_date(date_breaks = "2 weeks", date_labels = "%b %d") +
  labs(title = "Flights per week, spring 2020",
       x = "Week starting",
       y = "Flights")

### Your turn

**Q10.** Create `monthly`, with one row for each **month** in the data (5 years x 12 months = 60 rows) and two columns: `month_start` (the first day of the month, as a date) and `flights` (the number of flights that month).

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q10.R")

**Q11.** Make a **line graph** of your `monthly` table, with `month_start` on the x-axis and `flights` on the y-axis. Use `scale_x_date()` to put a tick mark and a label (just the year, like 2019) once per year. Add a **title** and plain-English axis labels. Save it as `q11.plot`.

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q11.R")

---
## 5. Making categories from dates: before, during, after

A big reason to work with dates is to compare **before and after** something happened. To do that you need a column saying which period each row belongs to. `case_when()` builds a new column from several conditions. R checks them **from top to bottom** and uses the first one that is `TRUE`. A final `TRUE ~ "something"` catches everything left over.

In [ ]:
events %>%
  mutate(date = ymd(date_text),
         stage = case_when(year(date) == 2020 ~ "first year",
                           year(date) == 2021 ~ "second year",
                           TRUE               ~ "later"))

(It's the same idea as the `ifelse()` you've used, but it handles more than two outcomes.)

Now we'll split the flight data into three periods using two real dates. Reasonable people could draw these lines in different places, and a researcher has to explain and defend choices like these:

- **`"pre"`**: before **March 19, 2020** (the day California's stay-at-home order began)
- **`"during"`**: from March 19, 2020 up to (but not including) **June 15, 2021** (the day California fully reopened)
- **`"post"`**: June 15, 2021 and everything after

### Your turn

**Q12.** Create `flights.period`, which is `flights.dates` plus a new column `period` that is `"pre"`, `"during"`, or `"post"` according to the definitions above.

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q12.R")

**Q13.** How did the three periods differ? Create `period.summary`, with one row per `period` and three columns: `flights` (the number of flights), `cancel_rate` (the share of flights cancelled; `cancelled` is 1 or 0), and `avg_dep_delay` (the average of `dep_delay`). Cancelled flights have `NA` for `dep_delay`, so you'll need to deal with that.

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q13.R")

**Q14.** Create `delay.year.airport`: the **average departure delay** (`avg_dep_delay`) for each combination of `year` and `airport`, using **only departing flights** (`direction == "departing"`). It should have 10 rows (5 years x 2 airports) and columns `year`, `airport`, and `avg_dep_delay`. Start from `flights.dates`. You'll need `filter()`, a date function, `group_by()`, and `summarize()`.

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q14.R")

---
## 6. Dates with times

`sched_departure` has a date **and** a time, like `"3/15/2019 14:30"`. To convert text like this, add the time pieces to the function name: `ymd_hm()`, `mdy_hm()`, `dmy_hm()` (hours and minutes), or `ymd_hms()` etc. (hours, minutes, and seconds). The result is a **date-time**. Then `hour()` and `minute()` pull out the pieces, just like `year()` and `month()` did. (R labels date-times with a time zone, "UTC" by default. You can ignore that here, since we only care about the clock time.)

In [ ]:
ymd_hm("2020-03-19 14:30")
mdy_hm("3/19/2020 14:30")
mdy_hm("3/19/2020 2:30 PM")      # AM/PM works too

t <- mdy_hm("3/19/2020 14:30")
hour(t)
minute(t)

Here is how you could count **departing** flights by the hour of the day they were scheduled to leave:

In [ ]:
flights.dates %>%
  filter(direction == "departing") %>%
  mutate(hour = hour(mdy_hm(sched_departure))) %>%
  group_by(hour) %>%
  summarize(flights = n())

### Your turn

**Q15.** Create `flights.times`, which is `flights.dates` plus two new columns: `sched_dep`, the `sched_departure` text converted to a date-time, and `dep_hour`, the **hour** (0 to 23) of that date-time.

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q15.R")

### Challenge

**Q16.** Put it all together. Make a **line graph** of the number of flights per **month**, with **one line for each airport** (colored by `airport`) and two **dashed vertical lines**: one at March 19, 2020 and one at June 15, 2021. Use `theme_bw()`, put the years on the x-axis with `scale_x_date()`, and add a title and plain-English axis and legend labels. Save it as `q16.plot`.

*Hints: First count flights for each combination of month and airport (group by two things), then pipe that into `ggplot()`. `geom_vline(xintercept = ymd("2020-03-19"), linetype = "dashed")` draws a vertical line at a date.*

In [ ]:
# Your code here

In [ ]:
. = ottr::check("tests/q16.R")

---
## What you learned

- `ymd()`, `mdy()`, `dmy()`: turn text into dates (the letters match the order of the text); `make_date()` builds a date from separate year, month, and day columns
- `year()`, `month()`, `day()`, `wday()`, `yday()`: pull pieces out of a date (`label = TRUE` gives names)
- Date math: subtract dates to get days apart, add `days()`, `weeks()`, `months()`, `years()`, and use `today()`
- Comparing dates with `<`, `>=`, `between()` to keep a date range
- `floor_date()`: round dates down to the week, month, or year so you can group by them
- `case_when()`: make categories like before / during / after from dates
- `mdy_hm()`, `ymd_hms()`, `hour()`, `minute()`: dates with times
- `geom_line()`, `scale_x_date()`, and `geom_vline()`: graph things over time

Time is everywhere in political science: election cycles, term limits, the days before a vote, before-and-after comparisons. Now you can work with it.